# MCTS Rollout — Caro 11×11

Notebook thí nghiệm cho tác tử MCTS (tìm kiếm cây Monte Carlo kết hợp rollout ngẫu nhiên).
Mọi thuật toán được import từ `agents/mcts_agent.py`; notebook chỉ tổ chức thí nghiệm và trình bày kết quả.

**Trạng thái hiện tại — Giai đoạn 3:** mục 1–6 (trạng thái, cây, bốn bước MCTS, năng lực quyết định) từ Giai đoạn 1–2;
mục 7–10 bổ sung khảo sát ngân sách `fixed` / `time`, thi đấu thử với `RandomAgent`, phân tích thất bại cùng biến thể có tên
`tactical`, tổng kết và hướng dẫn chạy lại.

Chạy notebook bằng **Restart Kernel & Run All** từ kernel `.venv`. Toàn bộ notebook mất khoảng 20–25 phút trên CPU ghi ở mục 2
(phần lớn ở mục 8–9 vì chế độ `time` dùng 2 giây cho mỗi nước của MCTS).


In [1]:
from pathlib import Path
import os

# Chuyển về thư mục Caro để import module chung từ vị trí notebook.
if Path.cwd().name == 'notebooks' and (Path.cwd().parent / 'core').is_dir():
    os.chdir(Path.cwd().parent)
if not (Path.cwd() / 'core' / 'game_engine.py').is_file():
    raise RuntimeError('Hãy mở notebook từ thư mục caro-11x11 hoặc notebooks')

import platform
import sys

import numpy as np
import pandas as pd

from agents.mcts_agent import (DEFAULT_EXPLORATION, DEFAULT_RADIUS, DEFAULT_TIME_MARGIN_S, MCTSAgent,
                               MCTSNode, MCTSState, outcome_value, uct_score)
from config.search_budget import SearchBudget
from core import CaroGame
from core.game_utils import candidate_actions


def left(df):
    """Căn trái nội dung và tiêu đề bảng để dễ đọc; chỉ dùng để hiển thị."""
    return df.style.set_properties(**{'text-align': 'left'}).set_table_styles(
        [{'selector': 'th', 'props': [('text-align', 'left')]}])

## 1. Câu hỏi nghiên cứu và giới hạn

**Mục tiêu:** xác định câu hỏi mà toàn bộ notebook trả lời và các giới hạn đã biết trước khi đo.

**Câu hỏi nghiên cứu:** MCTS thay đổi chất lượng nước đi và chi phí tính toán như thế nào khi tăng số lượt mô phỏng
(100 / 500 / 1.000 ở chế độ `fixed`) và khi bị giới hạn 2,0 giây/nước (chế độ `time`)?

**Giới hạn đã biết trước khi đo:**
- **Lọc ứng viên:** cây chỉ mở rộng các ô trống trong bán kính 2 quanh quân đã có (`candidate_actions`).
  Đây là phép rút gọn không gian tìm kiếm, không phải tập nước hợp lệ đầy đủ; nước tốt nằm xa hơn sẽ bị bỏ qua.
- **Chính sách rollout:** ngẫu nhiên, không có heuristic chiến thuật. Rollout ngẫu nhiên trên bàn 11×11 thường dài
  và nhiễu, nên MCTS có thể bỏ sót đe dọa ở ngân sách nhỏ. Chính sách cụ thể được chốt và đo ở Giai đoạn 2.
- **Thời gian:** cài đặt bằng Python thuần trên CPU; số mô phỏng đạt được trong 2,0 giây phụ thuộc phần cứng
  và sẽ được ghi nhận thực tế, không giả định trước.

Ô dưới đây đo quy mô của phép lọc ứng viên trên vài thế cờ tạo bằng engine.

**Kỳ vọng trước khi chạy (tính tay):**
- Bàn trống: 121 nước hợp lệ, 1 ứng viên (ô tâm (5, 5) theo quy ước của `candidate_actions`).
- 1 quân ở tâm: 120 nước hợp lệ; ứng viên là ô vuông 5×5 quanh quân trừ ô có quân = 25 − 1 = **24**.
- 1 quân ở góc (0, 0): ô vuông bị cắt ở mép còn 3×3 = 9 ô, trừ ô có quân = **8**.
- 4 quân giữa bàn: các ô vuông 5×5 chồng lên nhau nên ứng viên nhiều hơn 24 nhưng ít hơn 4 × 24.

In [2]:
def play_sequence(moves, size=11, k=5):
    """Tạo thế cờ bằng engine qua chuỗi nước hợp lệ (engine không có API nạp bàn cờ)."""
    game = CaroGame(size, k)
    for move in moves:
        game.step(move)
    return game

positions = {
    'bàn trống': [],
    '1 quân ở tâm': [(5, 5)],
    '4 quân giữa bàn': [(5, 5), (5, 6), (6, 5), (4, 4)],
    '1 quân ở góc': [(0, 0)],
}
rows = []
for name, moves in positions.items():
    game = play_sequence(moves)
    rows.append({'thế cờ': name, 'số quân': game.move_count,
                 'nước hợp lệ': len(game.legal_actions()),
                 f'ứng viên (r={DEFAULT_RADIUS})': len(candidate_actions(game.board, DEFAULT_RADIUS))})
left(pd.DataFrame(rows))

,thế cờ,số quân,nước hợp lệ,ứng viên (r=2)
0,bàn trống,0,121,1
1,1 quân ở tâm,1,120,24
2,4 quân giữa bàn,4,117,40
3,1 quân ở góc,1,120,8


**Quan sát so với kỳ vọng:** số liệu khớp phép tính tay ở cả bốn thế cờ (1 / 24 / 8 / 40).

**Nhận xét:** ở đầu ván, bán kính 2 thu hẹp nhánh từ khoảng 120 nước hợp lệ xuống vài chục ứng viên
(và chỉ 1 ứng viên khi bàn trống). Đây là lý do cây MCTS có thể đi sâu hơn với cùng ngân sách,
nhưng cũng là giới hạn phải công bố khi phân tích các ván thua.

## 2. Môi trường và cấu hình

**Mục tiêu:** ghi toàn bộ cấu hình tại một chỗ để mọi thí nghiệm sau dùng chung và có thể tái lập.

Các tham số dưới đây là cấu hình khởi đầu, chưa phải cấu hình tối ưu thực nghiệm.
Khi một thí nghiệm thay đổi tham số, cấu hình cụ thể sẽ được ghi kèm thí nghiệm đó.

**Định nghĩa thống kê** (chốt trước khi đo, áp dụng từ Giai đoạn 2 trong `agent.stats`, reset mỗi lượt):
| Chỉ số | Cách đếm |
| :--- | :--- |
| `simulations` | Số mô phỏng **hoàn thành**: rollout chạy tới trạng thái kết thúc và đã lan truyền ngược. Mô phỏng bị bỏ dở vì hết thời gian không được đếm và không cập nhật reward. |
| `nodes_visited` | Số nút **mới được tạo** trong cây ở lượt đó (không tính gốc). |
| `pruned_nodes` | Luôn 0: MCTS không cắt tỉa nhánh. Phép lọc ứng viên bán kính 2 không được tính là cắt tỉa. |
| `thinking_time` | Do `BaseAgent.choose_action` đo, đơn vị giây. |

In [3]:
SEED = 42                          # seed cho mọi yếu tố ngẫu nhiên
RADIUS = DEFAULT_RADIUS            # bán kính ứng viên
EXPLORATION = DEFAULT_EXPLORATION  # hệ số khám phá c của UCT (điểm xuất phát, chưa hiệu chỉnh)
ROLLOUT_POLICY = 'ngẫu nhiên đều trên toàn bộ ô trống, hai bên luân phiên, chơi tới khi kết thúc (không cắt ngắn)'
TIME_MARGIN_S = DEFAULT_TIME_MARGIN_S  # chế độ time: dừng ở deadline − 0,1 s (1,9 s trên hạn 2,0 s)
BUDGET_FIXED = SearchBudget(mode='fixed', simulations=1000)
BUDGET_TIME = SearchBudget(mode='time', time_limit_s=2.0)
FIXED_SURVEY = (100, 500, 1000)    # các mức khảo sát ở Giai đoạn 3

def cpu_name():
    """Tên CPU đầy đủ; trên Windows đọc từ registry, nơi khác dùng platform."""
    if sys.platform == 'win32':
        import winreg
        key = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r'HARDWARE\DESCRIPTION\System\CentralProcessor\0')
        return winreg.QueryValueEx(key, 'ProcessorNameString')[0].strip()
    return platform.processor() or platform.machine()

try:
    import torch
    torch_info = f'{torch.__version__} (CUDA: {torch.cuda.is_available()})'
except ImportError:
    torch_info = 'chưa cài'

config = {
    'seed': SEED, 'bán kính ứng viên': RADIUS, 'hệ số khám phá c': round(EXPLORATION, 4),
    'chính sách rollout': ROLLOUT_POLICY,
    'biên dừng chế độ time': f'{TIME_MARGIN_S} giây trước deadline',
    'quyết định cuối': 'con có N lớn nhất → Q/N cao hơn → ngẫu nhiên theo seed',
    'ngân sách fixed': f'{BUDGET_FIXED.simulations} mô phỏng/nước',
    'ngân sách time': f'{BUDGET_TIME.time_limit_s} giây/nước',
    'mức khảo sát fixed': FIXED_SURVEY,
    'Python': sys.version.split()[0], 'NumPy': np.__version__, 'pandas': pd.__version__,
    'PyTorch': torch_info, 'hệ điều hành': platform.platform(),
    'CPU': cpu_name(),
}
left(pd.Series(config, name='giá trị').to_frame())

,giá trị
seed,42
bán kính ứng viên,2
hệ số khám phá c,1.414200
chính sách rollout,"ngẫu nhiên đều trên toàn bộ ô trống, hai bên luân phiên, chơi tới khi kết thúc (không cắt ngắn)"
biên dừng chế độ time,0.1 giây trước deadline
quyết định cuối,con có N lớn nhất → Q/N cao hơn → ngẫu nhiên theo seed
ngân sách fixed,1000 mô phỏng/nước
ngân sách time,2.0 giây/nước
mức khảo sát fixed,"(100, 500, 1000)"
Python,3.11.9


**Nhận xét:** MCTS không dùng PyTorch/GPU; thông tin GPU chỉ ghi lại để đối chiếu môi trường chung của nhóm.
Thời gian chạy của MCTS phụ thuộc CPU nên mọi số liệu thời gian sau này phải đi kèm dòng `CPU` ở trên.

## 3. Kiểm tra trạng thái mô phỏng

**Mục tiêu:** minh họa `MCTSState` trước/sau nước đi và đối chiếu với engine `CaroGame`.

Engine không có API nạp bàn cờ tùy ý hay đi lùi, còn agent chỉ nhận `board` và `player`.
Vì vậy MCTS tự dựng trạng thái bằng `MCTSState.from_board(board, player)` (luôn sao chép bàn cờ)
và đi nước bằng luật giống engine. Tính đúng đắn được kiểm tra bằng cách chạy **cùng chuỗi nước hợp lệ**
trên cả hai và so sánh từng bước. Bộ unit test `tests/test_mcts_agent.py` kiểm tra chặt chẽ việc này;
notebook chỉ minh họa.

**Kỳ vọng trước khi chạy:** ở mọi bước, bàn cờ, người đến lượt, số nước, nước cuối và kết quả của hai bên trùng nhau.
Tại trạng thái kết thúc, `current_player` vẫn là người vừa đi nước cuối (đúng quy ước engine).

In [4]:
SYMBOLS = {0: '.', 1: 'X', -1: 'O'}

def board_text(board):
    """Chuỗi hiển thị bàn cờ; chỉ để minh họa, không thuộc thuật toán."""
    size = board.shape[0]
    head = '   ' + ' '.join(f'{c:>2}' for c in range(size))
    body = [f'{r:>2} ' + ' '.join(f'{SYMBOLS[int(v)]:>2}' for v in row) for r, row in enumerate(board)]
    return '\n'.join([head] + body)

def compare(state, game):
    return {
        'bàn cờ khớp': bool(np.array_equal(state.board, game.board)),
        'lượt khớp': state.current_player == game.current_player,
        'số nước khớp': state.move_count == game.move_count,
        'nước cuối khớp': state.last_action == game.last_action,
        'kết quả khớp': state.result() == game.result(),
    }

# Ván ngắn: X thắng hàng ngang ở nước thứ 9.
moves = [(5, 3), (6, 3), (5, 4), (6, 4), (5, 5), (6, 5), (5, 6), (4, 4), (5, 7)]
game = CaroGame()
state = MCTSState.from_board(game.board, game.current_player)
rows = []
for ply, move in enumerate(moves, start=1):
    game.step(move)
    state.step(move)
    rows.append({'nước': ply, 'tọa độ': move, 'người đi': SYMBOLS[int(state.board[move])],
                 'kết quả (góc X)': 'chưa kết thúc' if state.result() is None else state.result(),
                 'current_player': state.current_player,
                 **compare(state, game)})
trace = pd.DataFrame(rows)
trace

,nước,tọa độ,người đi,kết quả (góc X),current_player,bàn cờ khớp,lượt khớp,số nước khớp,nước cuối khớp,kết quả khớp
0,1,"(5, 3)",X,chưa kết thúc,-1,True,True,True,True,True
1,2,"(6, 3)",O,chưa kết thúc,1,True,True,True,True,True
2,3,"(5, 4)",X,chưa kết thúc,-1,True,True,True,True,True
3,4,"(6, 4)",O,chưa kết thúc,1,True,True,True,True,True
4,5,"(5, 5)",X,chưa kết thúc,-1,True,True,True,True,True
5,6,"(6, 5)",O,chưa kết thúc,1,True,True,True,True,True
6,7,"(5, 6)",X,chưa kết thúc,-1,True,True,True,True,True
7,8,"(4, 4)",O,chưa kết thúc,1,True,True,True,True,True
8,9,"(5, 7)",X,1,1,True,True,True,True,True


In [5]:
check_cols = ['bàn cờ khớp', 'lượt khớp', 'số nước khớp', 'nước cuối khớp', 'kết quả khớp']
print('Mọi bước đều khớp engine:', bool(trace[check_cols].all().all()))
print()
print('Trước nước cuối (đến lượt X):')
before = play_sequence(moves[:-1])
print(board_text(before.board))
print()
print('Sau nước cuối (5, 7) — X nối 5 quân:')
print(board_text(state.board))

Mọi bước đều khớp engine: True

Trước nước cuối (đến lượt X):
    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  O  .  .  .  .  .  .
 5  .  .  .  X  X  X  X  .  .  .  .
 6  .  .  .  O  O  O  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

Sau nước cuối (5, 7) — X nối 5 quân:
    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  O  .  .  .  .  .  .
 5  .  .  .  X  X  X  X  X  .  .  .
 6  .  .  .  O  O  O  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .


In [6]:
# Dựng trạng thái từ đầu vào kiểu agent (chỉ có board và player) ở giữa ván.
mid = play_sequence(moves[:4])
original = mid.board.copy()
from_input = MCTSState.from_board(mid.board, mid.current_player)
child = from_input.play((5, 5))
pd.DataFrame([
    {'trạng thái': 'dựng từ board', 'lượt': SYMBOLS[from_input.current_player],
     'số nước': from_input.move_count, 'nước cuối': from_input.last_action, 'kết quả': from_input.result()},
    {'trạng thái': 'sau play((5,5))', 'lượt': SYMBOLS[child.current_player],
     'số nước': child.move_count, 'nước cuối': child.last_action, 'kết quả': child.result()},
]).assign(**{'board đầu vào không đổi': bool(np.array_equal(mid.board, original)),
             'trạng thái cha không đổi': from_input.board[5, 5] == 0})

,trạng thái,lượt,số nước,nước cuối,kết quả,board đầu vào không đổi,trạng thái cha không đổi
0,dựng từ board,X,4,None,None,True,True
1,"sau play((5,5))",O,5,"(5, 5)",None,True,True


**Nhận xét:**
- Trạng thái mô phỏng khớp engine ở mọi bước của ván minh họa, kể cả bước kết thúc.
- Khi dựng từ đầu vào của agent, `last_action` là `None` vì agent không được biết nước cuối;
  do đó `from_board` quét toàn bộ bàn cờ một lần để nhận ra chuỗi thắng hoặc bàn đầy có sẵn.
- `play()` trả về trạng thái mới; trạng thái cha và mảng `board` đầu vào không bị thay đổi.
- `from_board` tin vào `player` do bên gọi đưa vào, không suy lượt từ số quân.

## 4. Quan sát cấu trúc cây

**Mục tiêu:** trình bày nút gốc, các nút con, lượt thăm (N), giá trị (Q) và quy ước góc nhìn trên một thế cờ nhỏ.

**Phân định dữ liệu: luật cờ và tìm kiếm**

| Lớp | Trường | Thuộc về | Ý nghĩa |
| :--- | :--- | :--- | :--- |
| `MCTSState` | `board` | Luật cờ | Bản sao bàn cờ `int8` (0 trống, 1 X, -1 O) |
| `MCTSState` | `current_player` | Luật cờ | Người đến lượt; tại terminal là người vừa đi (như engine) |
| `MCTSState` | `last_action`, `move_count` | Luật cờ | Nước cuối (None nếu dựng từ bàn cờ) và số quân đã đi |
| `MCTSState` | `result()` | Luật cờ | None / 1 / -1 / 0 theo góc nhìn X |
| `MCTSNode` | `state` | Liên kết | Trạng thái luật cờ tại nút |
| `MCTSNode` | `parent`, `children` | Tìm kiếm | Liên kết cha/con trong cây |
| `MCTSNode` | `action` | Tìm kiếm | Nước dẫn từ cha tới nút (None ở gốc) |
| `MCTSNode` | `player_just_moved` | Tìm kiếm | Góc nhìn dùng để cộng Q |
| `MCTSNode` | `untried_actions`, `radius` | Tìm kiếm | Ứng viên chưa mở rộng (rỗng nếu terminal) |
| `MCTSNode` | `visits` (N), `value` (Q) | Tìm kiếm | Thống kê, khởi tạo N = 0, Q = 0 |

**Quy ước giá trị (giữ nguyên ở mọi giai đoạn):**
- Kết quả ván `result` theo góc nhìn X như engine: `1` X thắng, `-1` O thắng, `0` hòa.
- Mỗi nút lưu `player_just_moved` — người đã đi nước dẫn tới nút. Ở gốc chưa kết thúc, đó là đối thủ của
  người đến lượt; ở gốc đã kết thúc, đó là `current_player` (engine giữ người vừa đi tại terminal).
- Q của nút được cộng dồn theo góc nhìn `player_just_moved`: thắng `+1`, thua `-1`, hòa `0`
  (`outcome_value(result, player) = result × player`).
- Hệ quả: đứng ở nút cha, người đến lượt chính là `player_just_moved` của các nút con, nên Q/N của mỗi con
  đã là giá trị **theo góc nhìn của chính người đang chọn** — ở tầng của X lẫn tầng của O, không cần đổi dấu.
- Q/N chỉ là **thành phần khai thác**. Khi tìm kiếm (Giai đoạn 2), nhánh được chọn theo điểm
  UCT = Q/N + c·√(ln N_cha / N_con), có thêm **thành phần khám phá** ưu tiên nhánh ít được thăm;
  nước chính thức trả về sau tìm kiếm được chọn theo số lượt thăm N.

**Kỳ vọng trước khi chạy:**
- Bảng ví dụ giá trị: X thắng → X +1, O −1; O thắng → O +1, X −1; hòa → cả hai 0.
- Thế cờ X(5,5), O(5,6), X(4,4), đến lượt O: gốc có `player_just_moved = X`; con tầng 1 là nước của O
  nên `player_just_moved = O`; con tầng 2 là nước của X nên `player_just_moved = X`.
- Mọi nút vừa tạo có N = 0, Q = 0, Q/N = 0 (chưa có tìm kiếm).
- Bàn cờ của gốc và của con tầng 1 không bị thay đổi sau khi mở rộng tầng dưới.

In [7]:
examples = []
for result, label in [(1, 'X thắng'), (-1, 'O thắng'), (0, 'hòa')]:
    examples.append({'kết cục': label, 'result (góc X)': result,
                     'giá trị cho X': outcome_value(result, 1), 'giá trị cho O': outcome_value(result, -1)})
pd.DataFrame(examples)

,kết cục,result (góc X),giá trị cho X,giá trị cho O
0,X thắng,1,1.0,-1.0
1,O thắng,-1,-1.0,1.0
2,hòa,0,0.0,0.0


In [8]:
# Thế cờ nhỏ: X ở (5,5), O ở (5,6), X ở (4,4); đến lượt O. Dùng bán kính 1 để cây dễ quan sát.
small = play_sequence([(5, 5), (5, 6), (4, 4)])
root = MCTSNode.root(small.board, small.current_player, radius=1)
print(board_text(small.board))
print()
print('Người đến lượt ở gốc:', SYMBOLS[root.state.current_player])
print('player_just_moved của gốc:', SYMBOLS[root.player_just_moved])
print('Số ứng viên chưa mở rộng ở gốc:', len(root.untried_actions))

# Mở rộng thủ công 3 nước đầu tiên của gốc và 1 nước ở tầng kế tiếp để quan sát liên kết.
for action in list(root.untried_actions[:3]):
    root.expand(action)
first_child = root.children[next(iter(root.children))]
grandchild = first_child.expand(first_child.untried_actions[0])

def node_row(node, depth):
    return {'độ sâu': depth, 'nước dẫn tới': node.action,
            'player_just_moved': SYMBOLS[node.player_just_moved],
            'đến lượt': SYMBOLS[node.state.current_player] if not node.is_terminal() else '-',
            'N': node.visits, 'Q': node.value, 'Q/N': node.mean_value(),
            'số con': len(node.children), 'chưa mở rộng': len(node.untried_actions),
            'cha': node.parent.action if node.parent and node.parent.parent else ('gốc' if node.parent else '-')}

rows = [node_row(root, 0)]
for child in root.children.values():
    rows.append(node_row(child, 1))
    rows.extend(node_row(g, 2) for g in child.children.values())
pd.DataFrame(rows)

    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  X  .  .  .  .  .  .
 5  .  .  .  .  .  X  O  .  .  .  .
 6  .  .  .  .  .  .  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

Người đến lượt ở gốc: O
player_just_moved của gốc: X
Số ứng viên chưa mở rộng ở gốc: 14


,độ sâu,nước dẫn tới,player_just_moved,đến lượt,N,Q,Q/N,số con,chưa mở rộng,cha
0,0,None,X,O,0,0.0,0.0,3,11,-
1,1,"(3, 3)",O,X,0,0.0,0.0,1,17,gốc
2,2,"(2, 2)",X,O,0,0.0,0.0,0,22,"(3, 3)"
3,1,"(3, 4)",O,X,0,0.0,0.0,0,16,gốc
4,1,"(3, 5)",O,X,0,0.0,0.0,0,17,gốc


In [9]:
# Tính độc lập: mở rộng con không làm đổi bàn cờ của cha.
print('Bàn cờ gốc vẫn như đầu vào:', bool(np.array_equal(root.state.board, small.board)))
print('Bàn cờ con tầng 1 chưa có nước của tầng 2:',
      int(first_child.state.board[grandchild.action]) == 0)
print('Liên kết cha/con đúng:', grandchild.parent is first_child and first_child.parent is root)

Bàn cờ gốc vẫn như đầu vào: True
Bàn cờ con tầng 1 chưa có nước của tầng 2: True
Liên kết cha/con đúng: True


**Quan sát so với kỳ vọng:** khớp toàn bộ — bảng giá trị đúng như dự kiến, góc nhìn luân phiên X → O → X
theo tầng, mọi nút có N = 0, Q = 0, và ba dòng kiểm tra tính độc lập đều `True`.

**Nhận xét:**
- Gốc có `player_just_moved = X` vì đến lượt O; các con tầng 1 có `player_just_moved = O`,
  các con tầng 2 có `player_just_moved = X` — góc nhìn luân phiên đúng theo tầng.
- Mọi nút mới có N = 0, Q = 0 vì cây ở mục này được mở rộng thủ công, chưa chạy mô phỏng nào.
- Phân bố N/Q thực tế sau tìm kiếm được trình bày ở mục 5.

## 5. Theo dõi bốn bước MCTS

**Mục tiêu:** quan sát trực tiếp từ module bốn bước của mỗi mô phỏng — Lựa chọn, Mở rộng, Mô phỏng, Cập nhật ngược —
và trạng thái cây sau tìm kiếm. Dữ liệu lấy qua tham số `trace` của `MCTSAgent.search`; notebook không cài lại thuật toán.

**Điều kiện dừng của từng bước** (cài trong `agents/mcts_agent.py`):
| Bước | Hàm | Dừng khi |
| :--- | :--- | :--- |
| Lựa chọn | `select_child` / `uct_score` | Gặp nút terminal hoặc nút còn nước chưa mở rộng |
| Mở rộng | `MCTSNode.expand` | Thêm đúng 1 con (nước chưa thử chọn ngẫu nhiên theo seed); bỏ qua nếu nút terminal |
| Mô phỏng | `rollout` | Có chuỗi thắng hoặc bàn đầy; nút terminal lấy thẳng kết quả, không rollout; quá mốc dừng thì bỏ mô phỏng |
| Cập nhật ngược | `backpropagate` | Đã cập nhật tới gốc (N += 1, Q += kết quả × `player_just_moved` ở từng nút) |

Vòng tìm kiếm dừng khi đủ `budget.simulations` mô phỏng hoàn thành (`fixed`) hoặc tới deadline − 0,1 s (`time`).

**Thế cờ quan sát:** X(5,5), O(5,6), X(6,5), O(4,4); đến lượt X.

**Kỳ vọng trước khi chạy (12 mô phỏng đầu):**
- Gốc còn nhiều ứng viên chưa thử nên bước Lựa chọn dừng ngay ở gốc (đường lựa chọn rỗng) và mỗi mô phỏng mở rộng một con mới.
- Mỗi rollout kết thúc bằng X thắng, O thắng hoặc hòa; giá trị cộng vào nút mới (nước của X) = kết quả × (+1).
- Sau 12 mô phỏng: N(gốc) = 12, gốc có 12 con, mỗi con N = 1.

In [10]:
observe = play_sequence([(5, 5), (5, 6), (6, 5), (4, 4)])
print(board_text(observe.board))
RESULT_TEXT = {1: 'X thắng', -1: 'O thắng', 0: 'hòa'}

def trace_table(records):
    rows = []
    for r in records:
        rows.append({'mô phỏng': r['simulation'],
                     'đường lựa chọn': ' → '.join(map(str, r['selection_path'])) or '(dừng ở gốc)',
                     'nước mở rộng': r['expanded'] if r['expanded'] is not None else '(nút terminal)',
                     'số nước rollout': len(r['rollout_moves']),
                     'kết quả': RESULT_TEXT[r['result']],
                     'giá trị cộng cho X': outcome_value(r['result'], 1)})
    return pd.DataFrame(rows)

records = []
small_agent = MCTSAgent(seed=SEED, budget=SearchBudget(simulations=12), exploration=EXPLORATION, radius=RADIUS)
root12, _ = small_agent.search(observe.board, observe.current_player, trace=records.append)
print()
print('N(gốc) =', root12.visits, '| số con =', len(root12.children),
      '| ứng viên chưa mở rộng =', len(root12.untried_actions))
left(trace_table(records))

    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  O  .  .  .  .  .  .
 5  .  .  .  .  .  X  O  .  .  .  .
 6  .  .  .  .  .  X  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

N(gốc) = 12 | số con = 12 | ứng viên chưa mở rộng = 28


,mô phỏng,đường lựa chọn,nước mở rộng,số nước rollout,kết quả,giá trị cộng cho X
0,1,(dừng ở gốc),"(7, 4)",73,O thắng,-1.000000
1,2,(dừng ở gốc),"(5, 3)",45,O thắng,-1.000000
2,3,(dừng ở gốc),"(3, 4)",58,X thắng,1.000000
3,4,(dừng ở gốc),"(4, 5)",35,O thắng,-1.000000
4,5,(dừng ở gốc),"(3, 3)",76,X thắng,1.000000
5,6,(dừng ở gốc),"(8, 3)",61,O thắng,-1.000000
6,7,(dừng ở gốc),"(3, 7)",73,O thắng,-1.000000
7,8,(dừng ở gốc),"(2, 5)",38,X thắng,1.000000
8,9,(dừng ở gốc),"(6, 8)",63,O thắng,-1.000000
9,10,(dừng ở gốc),"(3, 6)",88,X thắng,1.000000


Tiếp tục với 1.000 mô phỏng (`fixed`) trên cùng thế cờ. Khi mọi con của gốc đã được mở rộng, bước Lựa chọn
đi xuống sâu hơn theo UCT. Bảng dưới thống kê **độ dài đường lựa chọn** (số nút đã đi qua trước khi mở rộng)
và 10 con của gốc có N lớn nhất.

**Kỳ vọng:** `simulations` = 1.000 = N(gốc) = tổng N của các con; độ dài đường lựa chọn tăng dần khi cây đầy dần;
nước được chọn là con có N lớn nhất.

In [11]:
records = []
agent = MCTSAgent(seed=SEED, budget=BUDGET_FIXED, exploration=EXPLORATION, radius=RADIUS)
root, fallback = agent.search(observe.board, observe.current_player, trace=records.append)
depths = pd.Series([len(r['selection_path']) for r in records])
print('simulations =', agent.stats['simulations'], '| nodes_visited =', agent.stats['nodes_visited'],
      '| N(gốc) =', root.visits, '| tổng N các con =', sum(c.visits for c in root.children.values()))
print('Phân bố độ dài đường lựa chọn (độ dài: số mô phỏng):', depths.value_counts().sort_index().to_dict())
print('Độ dài trung bình theo từng khối 200 mô phỏng:',
      [round(float(depths[i:i + 200].mean()), 2) for i in range(0, len(depths), 200)])

top = sorted(root.children.values(), key=lambda c: (-c.visits, -c.mean_value()))[:10]
left(pd.DataFrame([{'nước': c.action, 'N': c.visits, 'Q': c.value, 'Q/N (góc X)': round(c.mean_value(), 3),
                    'UCT tại gốc': round(uct_score(c, root.visits, EXPLORATION), 3)} for c in top]))

simulations = 1000 | nodes_visited = 1000 | N(gốc) = 1000 | tổng N các con = 1000
Phân bố độ dài đường lựa chọn (độ dài: số mô phỏng): {0: 40, 1: 948, 2: 12}
Độ dài trung bình theo từng khối 200 mô phỏng: [0.8, 1.0, 1.0, 1.0, 1.06]


,nước,N,Q,Q/N (góc X),UCT tại gốc
0,"(7, 6)",58,14.000000,0.241000,0.729000
1,"(7, 5)",45,7.000000,0.156000,0.710000
2,"(6, 8)",43,7.000000,0.163000,0.730000
3,"(2, 2)",43,7.000000,0.163000,0.730000
4,"(6, 4)",42,6.000000,0.143000,0.716000
5,"(3, 8)",39,5.000000,0.128000,0.723000
6,"(4, 7)",38,4.000000,0.105000,0.708000
7,"(5, 3)",36,4.000000,0.111000,0.731000
8,"(3, 7)",34,2.000000,0.059000,0.696000
9,"(3, 6)",34,2.000000,0.059000,0.696000


In [12]:
# Quyết định cuối do agent đưa ra trên cùng cấu hình và seed (đối chiếu với bảng trên).
decider = MCTSAgent(seed=SEED, budget=BUDGET_FIXED, exploration=EXPLORATION, radius=RADIUS)
chosen = decider.choose_action(observe.board, observe.current_player)
best = max(decider.last_root.children.values(), key=lambda c: c.visits)
print('Nước agent chọn:', chosen, '| N =', decider.last_root.children[chosen].visits,
      '| N lớn nhất trong các con =', best.visits)
print('Thống kê lượt:', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in decider.stats.items()})

Nước agent chọn: (7, 6) | N = 58 | N lớn nhất trong các con = 58
Thống kê lượt: {'thinking_time': 0.236, 'nodes_visited': 1000, 'simulations': 1000, 'pruned_nodes': 0}


**Khám phá và khai thác:** chạy lại 1.000 mô phỏng với các hệ số c khác nhau, giữ nguyên seed và thế cờ.

**Kỳ vọng:** c = 0 chỉ khai thác nên lượt thăm dồn vào ít nhánh; c lớn rải lượt thăm đều hơn.

In [13]:
rows = []
for c in (0.0, 0.5, EXPLORATION, 2 * EXPLORATION):
    a = MCTSAgent(seed=SEED, budget=BUDGET_FIXED, exploration=c, radius=RADIUS)
    move = a.choose_action(observe.board, observe.current_player)
    visits = sorted((ch.visits for ch in a.last_root.children.values()), reverse=True)
    rows.append({'c': round(c, 3), 'số con của gốc': len(visits),
                 'con có N ≥ 10': sum(v >= 10 for v in visits),
                 'tỷ lệ N của con nhiều nhất': round(visits[0] / a.last_root.visits, 3),
                 'nước chọn': move})
left(pd.DataFrame(rows))

,c,số con của gốc,con có N ≥ 10,tỷ lệ N của con nhiều nhất,nước chọn
0,0.000000,40,7,0.821000,"(6, 6)"
1,0.500000,40,21,0.166000,"(7, 6)"
2,1.414000,40,38,0.058000,"(7, 6)"
3,2.828000,40,40,0.035000,"(5, 4)"


In [14]:
# Tái lập: cùng seed và ngân sách fixed cho cùng nước đi và cùng phân bố lượt thăm.
runs = []
for _ in range(2):
    a = MCTSAgent(seed=SEED, budget=BUDGET_FIXED)
    move = a.choose_action(observe.board, observe.current_player)
    runs.append((move, {k: v.visits for k, v in a.last_root.children.items()}))
print('Hai lần chạy cùng seed giống hệt nhau:', runs[0] == runs[1], '| nước:', runs[0][0])

Hai lần chạy cùng seed giống hệt nhau: True | nước: (7, 6)


**Quan sát so với kỳ vọng** (seed 42, số liệu của lần chạy này):
- 12 mô phỏng đầu đúng như dự kiến: cả 12 dừng lựa chọn ở gốc, mỗi mô phỏng mở rộng một con mới; N(gốc) = 12,
  12 con mỗi con N = 1; giá trị cộng cho X là +1 khi rollout X thắng và −1 khi O thắng. Rollout dài 35–104 nước.
- Với 1.000 mô phỏng: `simulations` = N(gốc) = tổng N các con = 1.000, khớp định nghĩa thống kê ở mục 2.
  `nodes_visited` = 1.000 vì mỗi mô phỏng mở rộng đúng một nút (chưa gặp nút terminal trên cây).
- Nước agent chọn trùng con có N lớn nhất trong bảng; hai lần chạy cùng seed cho cùng nước và cùng phân bố lượt thăm.
- c = 0 dồn 82% lượt thăm vào một nhánh; c = √2 rải lượt thăm (38/40 con có N ≥ 10, con nhiều nhất chỉ chiếm 5,8%).

**Nhận xét:**
- **Cây rất nông:** gốc có 40 ứng viên nên 40 mô phỏng đầu chỉ để mở rộng gốc; trong 1.000 mô phỏng chỉ 12 lần
  bước Lựa chọn đi xuống tới độ sâu 2. Với ngân sách này MCTS gần như chỉ đánh giá **một nước** bằng rollout ngẫu nhiên,
  nên chất lượng phụ thuộc mạnh vào độ nhiễu của rollout.
- Q/N của các con ở gốc chênh nhau nhỏ (khoảng 0,06–0,24) trong khi thành phần khám phá với c = √2 còn lớn (khoảng 0,49–0,64 ở 10 con đầu),
  nên UCT vẫn đang rải lượt thăm; nước được chọn chưa tách biệt rõ so với các nước khác.
- Ảnh hưởng của c tới chất lượng nước đi chưa được đo — chỉ quan sát phân bố lượt thăm; khảo sát thuộc Giai đoạn 3.

## 6. Kiểm tra năng lực quyết định

**Mục tiêu:** đo khả năng chọn đúng nước ở các thế chiến thuật cố định, qua 20 seed (0–19) và ba mức ngân sách
`fixed` 100 / 500 / 1.000 mô phỏng. Đây là phép đo, không phải yêu cầu MCTS ngẫu nhiên luôn đúng;
mọi seed thất bại đều được ghi lại.

| Thế cờ | Mô tả | Nước đúng (tính tay) |
| :--- | :--- | :--- |
| `thang_ngay` | X có 4 quân liền hở hai đầu ở hàng 5, đến lượt X | (5, 1) hoặc (5, 6): thắng ngay |
| `chan_bon` | O có 4 quân liền ở hàng 5, đầu trái bị X chặn; đến lượt X | (5, 5): ô duy nhất chặn được O |
| `thang_truoc_chan` | Cả X (hàng 3) và O (hàng 7) cùng có 4 quân hở; đến lượt X | (3, 0) hoặc (3, 5): thắng ngay thay vì chặn |

**Kỳ vọng trước khi chạy:** `thang_ngay` và `thang_truoc_chan` có nước thắng tức thì luôn cho giá trị +1 nên được chọn
ổn định khi ngân sách đủ lớn. `chan_bon` khó hơn với rollout ngẫu nhiên: sau đa số nước của X, O đi ngẫu nhiên
trong rollout hiếm khi hoàn thành chuỗi 5, nên nước chặn không nổi bật so với các nước khác.

In [15]:
import time

TACTICS = {
    'thang_ngay': ([(5, 2), (0, 0), (5, 3), (0, 2), (5, 4), (0, 4), (5, 5), (0, 6)], {(5, 1), (5, 6)}),
    'chan_bon': ([(5, 0), (5, 1), (0, 0), (5, 2), (0, 2), (5, 3), (10, 10), (5, 4)], {(5, 5)}),
    'thang_truoc_chan': ([(3, 1), (7, 1), (3, 2), (7, 2), (3, 3), (7, 3), (3, 4), (7, 4)], {(3, 0), (3, 5)}),
}
SEEDS = range(20)
for name, (moves, good) in TACTICS.items():
    g = play_sequence(moves)
    print(f'{name} — đến lượt {SYMBOLS[g.current_player]}, nước đúng {sorted(good)}')
    print(board_text(g.board))
    print()

thang_ngay — đến lượt X, nước đúng [(5, 1), (5, 6)]
    0  1  2  3  4  5  6  7  8  9 10
 0  O  .  O  .  O  .  O  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  .  .  .  .  .  .  .
 5  .  .  X  X  X  X  .  .  .  .  .
 6  .  .  .  .  .  .  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

chan_bon — đến lượt X, nước đúng [(5, 5)]
    0  1  2  3  4  5  6  7  8  9 10
 0  X  .  X  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  .  .  .  .  .  .  .
 5  X  O  O  O  O  .  .  .  .  .  .
 6  .  .  .  .  .  .  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  X

thang_truoc_chan — đến lượt X, nước đúng

In [16]:
rows, failures = [], []
for name, (moves, good) in TACTICS.items():
    g = play_sequence(moves)
    for sims in FIXED_SURVEY:
        picks, start = [], time.perf_counter()
        for seed in SEEDS:
            a = MCTSAgent(seed=seed, budget=SearchBudget(simulations=sims), exploration=EXPLORATION, radius=RADIUS)
            picks.append(a.choose_action(g.board, g.current_player))
        elapsed = (time.perf_counter() - start) / len(SEEDS)
        failed = [s for s, p in zip(SEEDS, picks) if p not in good]
        wrong = pd.Series([p for p in picks if p not in good], dtype=object).value_counts().head(3)
        rows.append({'thế cờ': name, 'mô phỏng': sims, 'đúng / 20 seed': f'{20 - len(failed)}/20',
                     'tỷ lệ đúng': round(1 - len(failed) / 20, 2),
                     'nước sai hay gặp': ', '.join(f'{k}×{v}' for k, v in wrong.items()) or '-',
                     'giây/nước (TB)': round(elapsed, 3)})
        failures.append({'thế cờ': name, 'mô phỏng': sims, 'seed thất bại': failed})
tactic_table = pd.DataFrame(rows)
left(tactic_table)

,thế cờ,mô phỏng,đúng / 20 seed,tỷ lệ đúng,nước sai hay gặp,giây/nước (TB)
0,thang_ngay,100,1/20,0.050000,"(4, 7)×3, (0, 5)×1, (1, 6)×1",0.022000
1,thang_ngay,500,19/20,0.950000,"(1, 3)×1",0.101000
2,thang_ngay,1000,20/20,1.000000,-,0.198000
3,chan_bon,100,0/20,0.000000,"(8, 9)×2, (7, 1)×2, (6, 0)×2",0.022000
4,chan_bon,500,1/20,0.050000,"(6, 6)×2, (7, 4)×2, (6, 1)×2",0.120000
5,chan_bon,1000,7/20,0.350000,"(4, 3)×2, (7, 6)×2, (0, 3)×1",0.228000
6,thang_truoc_chan,100,4/20,0.200000,"(4, 1)×2, (8, 6)×1, (2, 2)×1",0.019000
7,thang_truoc_chan,500,20/20,1.000000,-,0.074000
8,thang_truoc_chan,1000,20/20,1.000000,-,0.116000


In [17]:
left(pd.DataFrame(failures))

,thế cờ,mô phỏng,seed thất bại
0,thang_ngay,100,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19]"
1,thang_ngay,500,[13]
2,thang_ngay,1000,[]
3,chan_bon,100,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]"
4,chan_bon,500,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19]"
5,chan_bon,1000,"[1, 3, 4, 5, 8, 9, 10, 11, 12, 14, 17, 18, 19]"
6,thang_truoc_chan,100,"[1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 19]"
7,thang_truoc_chan,500,[]
8,thang_truoc_chan,1000,[]


**Quan sát so với kỳ vọng** (20 seed mỗi cấu hình):
| Thế cờ | 100 | 500 | 1.000 |
| :--- | :---: | :---: | :---: |
| `thang_ngay` | 1/20 | 19/20 | 20/20 |
| `chan_bon` | 0/20 | 1/20 | 7/20 |
| `thang_truoc_chan` | 4/20 | 20/20 | 20/20 |

**Phân tích:**
- **Thắng ngay** được chọn ổn định từ 500 mô phỏng: nút con là terminal nên mỗi lượt thăm luôn cộng +1, Q/N = 1 tuyệt đối.
  Ở 100 mô phỏng gần như luôn sai vì gốc có 59 ứng viên — mỗi con chỉ được thăm khoảng 1–2 lần, và vì X đã có
  4 quân hở, phần lớn nước khác của X cũng thắng trong rollout (O đi ngẫu nhiên hiếm khi chặn), nên nhiều con cùng Q/N = 1.
- **Thắng trước khi chặn** cũng đạt 20/20 từ 500: nước thắng ngay vượt trội cả các nước chặn.
- **Chặn bốn** là điểm yếu rõ nhất: chỉ 7/20 ở 1.000 mô phỏng. Sau nước không chặn của X, O trong rollout đi ngẫu nhiên
  trên ~110 ô trống nên chỉ hoàn thành chuỗi 5 với xác suất nhỏ; vì vậy các nước không chặn vẫn có Q/N gần với nước chặn.
  Đây là hạn chế của rollout ngẫu nhiên đều đã nêu ở mục 1, không phải lỗi góc nhìn (mục 4–5 và bộ test xác nhận góc nhìn đúng).
- Thời gian trung bình ~0,2 giây/nước ở 1.000 mô phỏng trên máy ghi ở mục 2, nên chế độ `time` 2,0 giây chạy được
  nhiều mô phỏng hơn hẳn; khảo sát chính thức của chế độ `time` thuộc mục 7 (Giai đoạn 3).
- Hướng cải thiện để đánh giá ở Giai đoạn 3 (biến thể có tên riêng, không thay bản chuẩn): rollout có heuristic
  (ưu tiên nước thắng/chặn ngay) hoặc tăng ngân sách.

## 7. Khảo sát ngân sách `fixed` và `time`

**Mục tiêu:** đo tác động của ngân sách tính toán lên chất lượng nước đi và chi phí, theo từng vị trí và qua nhiều seed.

**Cấu hình đóng băng trước khi đo** (ô dưới): bộ 6 vị trí, danh sách seed, bán kính 2, hệ số c = √2, biên dừng 0,1 giây,
biến thể `standard`. Trong khảo sát `fixed` chỉ thay **số mô phỏng** (100 / 500 / 1.000); chế độ `time` dùng 2,0 giây/nước và
**không** dùng số mô phỏng làm điều kiện dừng phụ. Biến thể `tactical` chỉ xuất hiện ở mục 9, cũng trên đúng cấu hình này.


In [18]:
from functools import partial

from agents.mcts_agent import VARIANTS
from agents.random_agent import RandomAgent
from evaluation.match import play_match
from evaluation.metrics import MatchRecord, summarize_latencies, summarize_results

# ---- Cấu hình đóng băng trước khi khảo sát (không đổi trong các mục 7–9) ----
SURVEY_POSITIONS = {
    # tên: (chuỗi nước hợp lệ, tập nước đúng tính tay hoặc None nếu không có đáp án duy nhất)
    'thang_ngay': (TACTICS['thang_ngay'][0], TACTICS['thang_ngay'][1]),
    'chan_bon': (TACTICS['chan_bon'][0], TACTICS['chan_bon'][1]),
    'thang_truoc_chan': (TACTICS['thang_truoc_chan'][0], TACTICS['thang_truoc_chan'][1]),
    'chan_ba_mo': ([(0, 0), (5, 3), (0, 3), (5, 4), (0, 6), (5, 5)], {(5, 2), (5, 6)}),
    'tao_bon_mo': ([(5, 3), (0, 0), (5, 4), (0, 3), (5, 5), (0, 6)], {(5, 2), (5, 6)}),
    'giua_van': ([(5, 5), (5, 6), (6, 5), (4, 4), (6, 6), (4, 6), (4, 5), (6, 4)], None),
}
SURVEY_SEEDS = list(range(10))      # seed MCTS cho chế độ fixed
TIME_SEEDS = list(range(5))         # seed MCTS cho chế độ time (mỗi lần 2,0 giây)
TIME_LIMIT_S = BUDGET_TIME.time_limit_s

for name in ['chan_ba_mo', 'tao_bon_mo', 'giua_van']:
    moves, good = SURVEY_POSITIONS[name]
    g = play_sequence(moves)
    print(f"{name} — đến lượt {SYMBOLS[g.current_player]}, nước đúng {sorted(good) if good else 'không có đáp án duy nhất'}")
    print(board_text(g.board))
    print()

left(pd.DataFrame([
    {'tham số': 'vị trí', 'giá trị': ', '.join(SURVEY_POSITIONS)},
    {'tham số': 'seed fixed', 'giá trị': f'{SURVEY_SEEDS[0]}–{SURVEY_SEEDS[-1]} ({len(SURVEY_SEEDS)} seed)'},
    {'tham số': 'seed time', 'giá trị': f'{TIME_SEEDS[0]}–{TIME_SEEDS[-1]} ({len(TIME_SEEDS)} seed)'},
    {'tham số': 'mức fixed', 'giá trị': f'{FIXED_SURVEY} mô phỏng/nước'},
    {'tham số': 'chế độ time', 'giá trị': f'{TIME_LIMIT_S} giây/nước, dừng ở deadline − {TIME_MARGIN_S} giây'},
    {'tham số': 'bán kính, hệ số c', 'giá trị': f'{RADIUS}, {round(EXPLORATION, 4)}'},
    {'tham số': 'biến thể', 'giá trị': f'{VARIANTS[0]} (mục 7–8); {VARIANTS[1]} chỉ ở mục 9'},
]))

chan_ba_mo — đến lượt X, nước đúng [(5, 2), (5, 6)]
    0  1  2  3  4  5  6  7  8  9 10
 0  X  .  .  X  .  .  X  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  .  .  .  .  .  .  .
 5  .  .  .  O  O  O  .  .  .  .  .
 6  .  .  .  .  .  .  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

tao_bon_mo — đến lượt X, nước đúng [(5, 2), (5, 6)]
    0  1  2  3  4  5  6  7  8  9 10
 0  O  .  .  O  .  .  O  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  .  .  .  .  .  .  .
 5  .  .  .  X  X  X  .  .  .  .  .
 6  .  .  .  .  .  .  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

giua_van — đến lượt X, nước đú

,tham số,giá trị
0,vị trí,"thang_ngay, chan_bon, thang_truoc_chan, chan_ba_mo, tao_bon_mo, giua_van"
1,seed fixed,0–9 (10 seed)
2,seed time,0–4 (5 seed)
3,mức fixed,"(100, 500, 1000) mô phỏng/nước"
4,chế độ time,"2.0 giây/nước, dừng ở deadline − 0.1 giây"
5,"bán kính, hệ số c","2, 1.4142"
6,biến thể,standard (mục 7–8); tactical chỉ ở mục 9


Ba vị trí đầu lấy lại từ mục 6. Ba vị trí mới:

| Vị trí | Mô tả | Nước đúng (tính tay) |
| :--- | :--- | :--- |
| `chan_ba_mo` | O có ba quân liền hở hai đầu ở hàng 5; đến lượt X | (5, 2) hoặc (5, 6). Chặn xa hơn như (5, 1) vẫn để O đi (5, 6) thành bốn mở và thắng |
| `tao_bon_mo` | X có ba quân liền hở hai đầu; đến lượt X | (5, 2) hoặc (5, 6): tạo bốn mở, O không chặn được cả hai đầu |
| `giua_van` | Giữa ván, chưa bên nào có chuỗi ba | Không có đáp án duy nhất; chỉ đo chi phí và độ ổn định qua seed |

Hai vị trí mới cần **nhìn trước hai nước** (khác ba vị trí đầu chỉ cần một nước), nên khó hơn với MCTS.

**Chỉ số đo** (mỗi lần chọn nước là một dòng trong bảng thô `survey_fixed` / `survey_time`):
`đúng` (nước nằm trong tập tính tay), số nước khác nhau qua các seed (độ biến động của quyết định), `simulations`,
`nodes_visited` (nút mới), `thinking_time` và số lần vượt hạn 2,0 giây.

**Kỳ vọng trước khi chạy:** tỷ lệ đúng tăng theo ngân sách; `chan_bon` và hai vị trí hai nước khó hơn `thang_ngay`;
thời gian ở chế độ `fixed` tăng gần tuyến tính theo số mô phỏng; chế độ `time` chạy nhiều mô phỏng hơn hẳn 1.000.


In [19]:
def measure(variant, seeds, budgets):
    """Chạy MCTS trên mọi vị trí × seed × ngân sách; trả bảng thô, mỗi dòng là một lần chọn nước."""
    rows = []
    for name, (moves, good) in SURVEY_POSITIONS.items():
        g = play_sequence(moves)
        for budget in budgets:
            for seed in seeds:
                agent = MCTSAgent(seed=seed, budget=budget, exploration=EXPLORATION, radius=RADIUS,
                                  time_margin_s=TIME_MARGIN_S, variant=variant)
                action = agent.choose_action(g.board, g.current_player)
                rows.append({'biến thể': variant, 'vị trí': name, 'chế độ': budget.mode,
                             'ngân sách': budget.simulations if budget.mode == 'fixed' else f'{budget.time_limit_s} s',
                             'seed': seed, 'nước': action, 'đúng': None if good is None else action in good,
                             'mô phỏng': agent.stats['simulations'], 'nút mới': agent.stats['nodes_visited'],
                             'ứng viên bị lọc': agent.stats['pruned_nodes'],
                             'thời gian (s)': agent.stats['thinking_time']})
    return pd.DataFrame(rows)


def summarize_survey(raw):
    """Tổng hợp theo vị trí × ngân sách: tỷ lệ đúng, số nước khác nhau qua seed, trung bình ± độ lệch chuẩn."""
    out = []
    for (name, mode, budget), grp in raw.groupby(['vị trí', 'chế độ', 'ngân sách'], sort=False):
        known = grp['đúng'].dropna()
        out.append({'vị trí': name, 'chế độ': mode, 'ngân sách': budget, 'số lần': len(grp),
                    'đúng': f'{int(known.sum())}/{len(known)}' if len(known) else '-',
                    'số nước khác nhau': grp['nước'].nunique(),
                    'mô phỏng TB': round(grp['mô phỏng'].mean()),
                    'mô phỏng min–max': f"{grp['mô phỏng'].min()}–{grp['mô phỏng'].max()}",
                    'nút mới TB ± ĐLC': f"{grp['nút mới'].mean():.0f} ± {grp['nút mới'].std(ddof=0):.0f}",
                    'thời gian TB ± ĐLC (s)': f"{grp['thời gian (s)'].mean():.3f} ± {grp['thời gian (s)'].std(ddof=0):.3f}",
                    f'vượt {TIME_LIMIT_S} s': int((grp['thời gian (s)'] >= TIME_LIMIT_S).sum())})
    return pd.DataFrame(out)


fixed_budgets = [SearchBudget(mode='fixed', simulations=n) for n in FIXED_SURVEY]
start = time.perf_counter()
survey_fixed = measure('standard', SURVEY_SEEDS, fixed_budgets)
survey_time = measure('standard', TIME_SEEDS, [BUDGET_TIME])
print(f'Thời gian chạy khảo sát: {time.perf_counter() - start:.0f} giây')

# Kiểm tra toàn vẹn: đủ vị trí × ngân sách × seed; fixed chạy đúng số mô phỏng yêu cầu.
assert len(survey_fixed) == len(SURVEY_POSITIONS) * len(FIXED_SURVEY) * len(SURVEY_SEEDS)
assert len(survey_time) == len(SURVEY_POSITIONS) * len(TIME_SEEDS)
assert (survey_fixed['mô phỏng'] == survey_fixed['ngân sách']).all()
assert (survey_fixed['ứng viên bị lọc'] == 0).all() and (survey_time['ứng viên bị lọc'] == 0).all()
survey_standard = summarize_survey(pd.concat([survey_fixed, survey_time], ignore_index=True))
left(survey_standard)

Thời gian chạy khảo sát: 77 giây


,vị trí,chế độ,ngân sách,số lần,đúng,số nước khác nhau,mô phỏng TB,mô phỏng min–max,nút mới TB ± ĐLC,thời gian TB ± ĐLC (s),vượt 2.0 s
0,thang_ngay,fixed,100,10,0/10,10,100,100–100,98 ± 1,0.020 ± 0.001,0
1,thang_ngay,fixed,500,10,10/10,2,500,500–500,457 ± 3,0.101 ± 0.003,0
2,thang_ngay,fixed,1000,10,10/10,2,1000,1000–1000,852 ± 10,0.189 ± 0.003,0
3,chan_bon,fixed,100,10,0/10,9,100,100–100,100 ± 0,0.021 ± 0.001,0
4,chan_bon,fixed,500,10,0/10,10,500,500–500,500 ± 0,0.123 ± 0.041,0
5,chan_bon,fixed,1000,10,4/10,6,1000,1000–1000,1000 ± 0,0.225 ± 0.010,0
6,thang_truoc_chan,fixed,100,10,2/10,9,100,100–100,96 ± 1,0.019 ± 0.001,0
7,thang_truoc_chan,fixed,500,10,10/10,2,500,500–500,356 ± 13,0.075 ± 0.004,0
8,thang_truoc_chan,fixed,1000,10,10/10,2,1000,1000–1000,479 ± 27,0.121 ± 0.039,0
9,chan_ba_mo,fixed,100,10,0/10,8,100,100–100,100 ± 0,0.022 ± 0.001,0


**Quan sát so với kỳ vọng.** Kết quả `fixed` cố định theo seed; số mô phỏng và thời gian của `time` phụ thuộc máy.
- **Một nước:** `thang_ngay` và `thang_truoc_chan` đúng 10/10 từ 500 mô phỏng, nhưng gần như luôn sai ở 100 (0/10 và 2/10)
  vì mỗi ứng viên chỉ được thăm khoảng 1–2 lần. `chan_bon` khó hơn nhiều: 0 / 0 / 4 trên 10 ở chế độ `fixed`, nhưng 5/5 ở
  chế độ `time` với khoảng 8.000 mô phỏng, tức tăng ngân sách là đủ để nước chặn nổi bật.
- **Hai nước:** `tao_bon_mo` tăng dần 1 / 4 / 5 trên 10 và đạt 5/5 ở chế độ `time`. `chan_ba_mo` là vị trí khó nhất:
  0 / 1 / 1 trên 10 và vẫn chỉ 1/5 ở chế độ `time`. Muốn thấy nước chặn ba mở, cây phải đi qua nước của O (thành bốn mở) rồi
  nước của X, trong khi mỗi tầng có khoảng 50 ứng viên được chia đều lượt thăm.
- **Độ biến động qua seed:** số nước khác nhau giảm khi tăng ngân sách. Ở `giua_van`: 10 → 9 → 7 nước khác nhau trên 10 seed ở
  chế độ `fixed`, và chỉ 1 nước ở chế độ `time`.
- **Số nút mới** xấp xỉ số mô phỏng, trừ các vị trí có nước thắng ngay (`thang_ngay` 852, `thang_truoc_chan` 479 nút ở 1.000
  mô phỏng): mô phỏng đi vào nút con đã kết thúc không tạo nút mới.
- **Chế độ `time`:** gần như mọi lần đều dừng ở khoảng 1,9 giây theo biên an toàn, nhưng có **1 lần** (ở `tao_bon_mo`) thời gian
  suy nghĩ chạm mốc 2,0 giây. Biên 0,1 giây là cơ chế dừng hợp tác, không phải bảo đảm tuyệt đối: một lượt có thể bị trễ vì hệ
  điều hành hoặc bộ dọn rác của Python. Lần đo này gọi agent trực tiếp; trong 90 ván qua runner ở mục 8–9 không có timeout nào.
- **Số mô phỏng ở chế độ `time`** phụ thuộc mạnh vào vị trí: khoảng 7.000–8.000 ở vị trí thường, nhưng nhiều gấp nhiều lần (hàng
  chục nghìn) ở vị trí có nước thắng ngay, vì mô phỏng kết thúc ngay ở nút thắng thì không cần rollout.


In [20]:
# Gộp theo ngân sách (chỉ các vị trí có đáp án): tổng số lần đúng, mô phỏng và thời gian trung bình.
raw_standard = pd.concat([survey_fixed, survey_time], ignore_index=True)
known_raw = raw_standard[raw_standard['đúng'].notna()]
overview = known_raw.groupby('ngân sách', sort=False).agg(
    lan_dung=('đúng', 'sum'), so_lan=('đúng', 'size'),
    mo_phong_tb=('mô phỏng', 'mean'), thoi_gian_tb=('thời gian (s)', 'mean')).reset_index()
overview['đúng'] = overview['lan_dung'].astype(int).astype(str) + '/' + overview['so_lan'].astype(str)
overview['mô phỏng TB'] = overview['mo_phong_tb'].round().astype(int)
overview['thời gian TB (s)'] = overview['thoi_gian_tb'].map(lambda v: f'{v:.3f}')
left(overview[['ngân sách', 'đúng', 'mô phỏng TB', 'thời gian TB (s)']])

,ngân sách,đúng,mô phỏng TB,thời gian TB (s)
0,100,3/50,100,0.021
1,500,25/50,500,0.108
2,1000,30/50,1000,0.200
3,2.0 s,21/25,35779,1.905


**Nhận xét tổng hợp** (5 vị trí có đáp án): tỷ lệ đúng tăng từ 3/50 (100 mô phỏng) lên 25/50 (500), 30/50 (1.000)
và 21/25 ở chế độ `time`. Bước từ 100 lên 500 cải thiện nhiều nhất; từ 500 lên 1.000 chỉ thêm 5 lần đúng vì các lỗi còn lại
(chặn bốn, chặn ba mở) cần nhiều mô phỏng hơn hẳn. Thời gian ở chế độ `fixed` tăng gần tuyến tính theo số mô phỏng, nên 1.000 mô
phỏng chỉ tốn khoảng một phần mười ngân sách 2 giây; chế độ `time` tận dụng phần còn lại và cho kết quả tốt nhất.


## 8. Thi đấu thử với `RandomAgent`

**Mục tiêu:** xác nhận MCTS chơi trọn ván qua runner chính thức `play_match` (mỗi agent một tiến trình, hạn cứng, ghi log
từng lượt) ở cả hai chế độ, cân bằng đi trước/đi sau.

**Cấu hình:**
- 10 cặp seed: MCTS dùng seed `s`, Random dùng seed `1000 + s`, `s` = 0–9. Mỗi cặp chơi 2 ván đổi bên, tức **20 ván mỗi chế độ**.
- `fixed`: 1.000 mô phỏng/nước, watchdog 30 giây/nước của runner. `time`: 2,0 giây/nước, runner dùng thẳng hạn 2,0 giây.
- W/D/L, điểm và độ trễ tính bằng `evaluation/metrics.py` (`summarize_results`, `summarize_latencies`); độ trễ là `elapsed_s`
  do runner đo (gồm cả giao tiếp giữa tiến trình). Ván xử thua do timeout hay lỗi được tính vào W/D/L và đếm riêng ở cột
  `lỗi/timeout`.
- Kiểm tra toàn vẹn trong ô: đủ 20 ván, mọi lượt của MCTS ghi đúng chế độ và giới hạn; ở `fixed` mỗi lượt đúng 1.000 mô phỏng.

Đây là nghiệm thu phát triển, **không phải giải đấu chính thức** (tối thiểu 100 ván/cặp, do nhóm trưởng điều phối).


In [21]:
MATCH_SEEDS = list(range(10))   # cặp seed: MCTS dùng seed s, Random dùng seed 1000 + s; mỗi cặp chơi 2 ván đổi bên


def run_series(label, factory_a, opponent_label, factory_b, budget, seeds):
    """Mỗi seed chơi 2 ván qua runner play_match (A cầm X rồi cầm O). Trả MatchRecord, bảng từng ván, log lượt của A."""
    records, games, a_logs = [], [], []
    for seed in seeds:
        for a_side in (1, -1):
            fa, fb = factory_a(seed), factory_b(seed)
            fx, fo = (fa, fb) if a_side == 1 else (fb, fa)
            name_x, name_o = (label, opponent_label) if a_side == 1 else (opponent_label, label)
            record, logs = play_match(fx, fo, agent_x=name_x, agent_o=name_o, budget=budget)
            records.append(record)
            mine = [log for log in logs if log['player'] == a_side]
            a_logs.extend(dict(log, seed=seed, side=a_side) for log in mine)
            outcome = record.winner * a_side
            games.append({'seed': seed, f'{label} cầm': SYMBOLS[a_side], 'số nước': len(logs),
                          'kết quả': {1: 'thắng', 0: 'hòa', -1: 'thua'}[outcome], 'lý do': record.reason})
    return records, pd.DataFrame(games), pd.DataFrame(a_logs)


def series_summary(label, records, a_logs, budget):
    """W/D/L, điểm và độ trễ tính bằng evaluation/metrics.py; lỗi và timeout báo riêng."""
    res = summarize_results(records, label)
    moves = a_logs[a_logs['status'] == 'move']
    lat = summarize_latencies(moves['elapsed_s'].tolist())
    sims = moves['stats'].map(lambda s: s['simulations'])
    return {'cấu hình': label, 'chế độ': budget.mode, 'số ván': res['total'], 'thắng': res['wins'],
            'hòa': res['draws'], 'thua': res['losses'], 'điểm': f"{res['score']:.3f}",
            'lỗi/timeout': sum(res['forfeits'].values()), 'số lượt': lat['count'],
            'độ trễ TB (s)': f"{lat['mean_s']:.3f}", 'độ trễ p95 (s)': f"{lat['p95_s']:.3f}",
            'mô phỏng/lượt TB': round(sims.mean()), 'mô phỏng/lượt min–max': f'{sims.min()}–{sims.max()}'}


def check_series(records, a_logs, budget, seeds):
    """Toàn vẹn: đủ số ván, W/D/L khớp, mọi lượt của A ghi đúng chế độ và giới hạn của cấu hình."""
    assert len(records) == 2 * len(seeds)
    assert set(a_logs['mode']) == {budget.mode}
    assert set(a_logs['limit_s']) == {budget.time_limit_s if budget.mode == 'time' else 30.0}
    if budget.mode == 'fixed':
        assert (a_logs['stats'].map(lambda s: s['simulations']) == budget.simulations).all()


mcts_fixed = lambda s: partial(MCTSAgent, seed=s, exploration=EXPLORATION, radius=RADIUS)
mcts_time = lambda s: partial(MCTSAgent, seed=s, exploration=EXPLORATION, radius=RADIUS, time_margin_s=TIME_MARGIN_S)
random_factory = lambda s: partial(RandomAgent, seed=1000 + s)

start = time.perf_counter()
rec_fixed, games_fixed, logs_fixed = run_series('MCTS fixed 1000', mcts_fixed, 'Ngẫu nhiên', random_factory,
                                                BUDGET_FIXED, MATCH_SEEDS)
check_series(rec_fixed, logs_fixed, BUDGET_FIXED, MATCH_SEEDS)
rec_time, games_time, logs_time = run_series('MCTS time 2,0 s', mcts_time, 'Ngẫu nhiên', random_factory,
                                             BUDGET_TIME, MATCH_SEEDS)
check_series(rec_time, logs_time, BUDGET_TIME, MATCH_SEEDS)
print(f'Thời gian chạy thi đấu: {time.perf_counter() - start:.0f} giây')
match_summary = pd.DataFrame([series_summary('MCTS fixed 1000', rec_fixed, logs_fixed, BUDGET_FIXED),
                              series_summary('MCTS time 2,0 s', rec_time, logs_time, BUDGET_TIME)])
left(match_summary)

Thời gian chạy thi đấu: 303 giây


,cấu hình,chế độ,số ván,thắng,hòa,thua,điểm,lỗi/timeout,số lượt,độ trễ TB (s),độ trễ p95 (s),mô phỏng/lượt TB,mô phỏng/lượt min–max
0,MCTS fixed 1000,fixed,20,20,0,0,1.000,0,212,0.250,0.277,1000,1000–1000
1,"MCTS time 2,0 s",time,20,20,0,0,1.000,0,122,1.901,1.901,11276,6570–48262


**Quan sát:** MCTS thắng **20/20** ván ở cả hai chế độ, không hòa, không thua và **không có timeout hay lỗi**.
- `fixed`: 212 lượt của MCTS, lượt nào cũng đúng 1.000 mô phỏng. Độ trễ do runner đo cao hơn `thinking_time` ở mục 7 một chút vì
  gồm cả giao tiếp giữa tiến trình, nhưng vẫn rất xa hạn 30 giây.
- `time`: độ trễ trung bình và p95 đều khoảng 1,9 giây, tức dừng đúng biên an toàn và còn khoảng 0,1 giây so với hạn cứng 2,0
  giây. Số mô phỏng mỗi lượt dao động rất rộng (khoảng 6.500 đến gần 50.000), cùng nguyên nhân ở mục 7: lượt có nước thắng
  ngay chạy được nhiều mô phỏng hơn hẳn.
- Ván ở chế độ `time` thường ngắn hơn (9–22 nước, đa số 9–11 nước, so với 10–31 nước ở `fixed`): ngân sách lớn hơn giúp MCTS
  kết thúc ván nhanh hơn.
- Random là đối thủ rất yếu nên kết quả này chỉ xác nhận tích hợp, thời gian và tính đúng của runner, không đo sức mạnh.


In [22]:
# Chi tiết từng ván (không bỏ ván nào); cột "số nước" tính cả hai bên.
left(pd.concat([games_fixed.assign(**{'cấu hình': 'MCTS fixed 1000'}).rename(columns={'MCTS fixed 1000 cầm': 'MCTS cầm'}),
                games_time.assign(**{'cấu hình': 'MCTS time 2,0 s'}).rename(columns={'MCTS time 2,0 s cầm': 'MCTS cầm'})],
               ignore_index=True)[['cấu hình', 'seed', 'MCTS cầm', 'số nước', 'kết quả', 'lý do']])

,cấu hình,seed,MCTS cầm,số nước,kết quả,lý do
0,MCTS fixed 1000,0,X,17,thắng,completed
1,MCTS fixed 1000,0,O,10,thắng,completed
2,MCTS fixed 1000,1,X,13,thắng,completed
3,MCTS fixed 1000,1,O,22,thắng,completed
4,MCTS fixed 1000,2,X,19,thắng,completed
5,MCTS fixed 1000,2,O,16,thắng,completed
6,MCTS fixed 1000,3,X,31,thắng,completed
7,MCTS fixed 1000,3,O,30,thắng,completed
8,MCTS fixed 1000,4,X,15,thắng,completed
9,MCTS fixed 1000,4,O,28,thắng,completed


Cả 40 ván đều có lý do kết thúc `completed`; không ván nào bị loại khỏi bảng.


## 9. Phân tích thất bại và biến thể `tactical`

**Các thất bại quan sát được ở mục 6–7:**
1. **Ngân sách nhỏ:** ở 100 mô phỏng, gốc có khoảng 50–60 ứng viên nên mỗi ứng viên chỉ được thăm 1–2 lần; ngay cả nước thắng ngay
   cũng không nổi bật.
2. **Chặn đe dọa một nước** (`chan_bon`): sau một nước không chặn của X, O trong rollout đi ngẫu nhiên trên khoảng 110 ô trống nên
   hiếm khi hoàn thành chuỗi năm; nước chặn chỉ nổi bật khi có hàng nghìn mô phỏng.
3. **Đe dọa hai nước** (`chan_ba_mo`): ngay cả 2 giây cũng chỉ đúng 1/5. Cây phải phát hiện nước thành bốn mở của O ở tầng 2,
   nhưng lượt thăm ở tầng đó chia đều cho khoảng 50 nước trả lời.
4. **Đe dọa kép:** khi đối thủ đã có hai ô thắng ngay thì không thuật toán nào chặn kịp; đây là giới hạn của thế cờ, không phải lỗi.

**Biến thể có tên `tactical`** (`MCTSAgent(variant='tactical')`), chỉ thay hai chỗ, mọi phần khác dùng chung với `standard`:
- **Ứng viên của mỗi nút:** bên đến lượt có ô thắng ngay thì chỉ giữ các ô đó; nếu không mà đối thủ có ô thắng ngay thì chỉ giữ ô
  chặn; ngược lại giữ ứng viên bán kính 2 như bản chuẩn (kỹ thuật *decisive / anti-decisive moves*). Số ứng viên bị bỏ được ghi
  vào `pruned_nodes`.
- **Rollout:** cùng luật đó ở mỗi nước, còn lại đi ngẫu nhiên như bản chuẩn.

Chỉ thay rollout mà giữ nguyên cách mở rộng thì chưa đủ: luật chiến thuật cần có ở cả bước mở rộng để nước trả lời bắt buộc
không bị chia lượt thăm với khoảng 50 nước khác. Ô dưới chạy lại đúng vị trí, seed và ngân sách của mục 7, chỉ đổi biến thể.


In [23]:
# Cùng vị trí, seed và ngân sách như mục 7; chỉ đổi biến thể standard → tactical.
start = time.perf_counter()
tac_fixed = measure('tactical', SURVEY_SEEDS, fixed_budgets)
tac_time = measure('tactical', TIME_SEEDS, [BUDGET_TIME])
print(f'Thời gian chạy khảo sát biến thể: {time.perf_counter() - start:.0f} giây')
assert len(tac_fixed) == len(survey_fixed) and len(tac_time) == len(survey_time)
assert (tac_fixed['mô phỏng'] == tac_fixed['ngân sách']).all()
raw_tactical = pd.concat([tac_fixed, tac_time], ignore_index=True)

def accuracy_table(raw):
    known = raw[raw['đúng'].notna()]
    return known.groupby(['vị trí', 'ngân sách'], sort=False)['đúng'].agg(
        lambda s: f'{int(s.sum())}/{len(s)}').unstack('ngân sách')

compare = pd.concat({'standard': accuracy_table(raw_standard), 'tactical': accuracy_table(raw_tactical)}, axis=1)
display(left(compare))
cost = pd.concat([raw_standard, raw_tactical]).groupby(['biến thể', 'ngân sách'], sort=False).agg(
    **{'mô phỏng TB': ('mô phỏng', 'mean'), 'thời gian TB (s)': ('thời gian (s)', 'mean'),
       'ứng viên bị lọc TB': ('ứng viên bị lọc', 'mean')}).reset_index()
cost['mô phỏng TB'] = cost['mô phỏng TB'].round().astype(int)
cost['thời gian TB (s)'] = cost['thời gian TB (s)'].map(lambda v: f'{v:.3f}')
cost['ứng viên bị lọc TB'] = cost['ứng viên bị lọc TB'].map(lambda v: f'{v:.1f}')
left(cost)

Thời gian chạy khảo sát biến thể: 107 giây


,biến thể,ngân sách,mô phỏng TB,thời gian TB (s),ứng viên bị lọc TB
0,standard,100,100,0.021,0.0
1,standard,500,500,0.111,0.0
2,standard,1000,1000,0.204,0.0
3,standard,2.0 s,31120,1.904,0.0
4,tactical,100,100,0.057,179.5
5,tactical,500,500,0.269,491.9
6,tactical,1000,1000,0.502,811.6
7,tactical,2.0 s,153455,1.900,1680.3


**Quan sát:**
- Ba vị trí **một nước** đạt 10/10 ngay từ 100 mô phỏng. Đây là tác dụng trực tiếp của luật lọc ứng viên ở gốc, không phải do tìm
  kiếm mạnh hơn; nó loại bỏ hẳn thất bại 1 và 2 ở trên.
- **Hai nước** mới là phần cho thấy tìm kiếm tốt hơn: `chan_ba_mo` tăng từ 0 / 1 / 1 lên 1 / 3 / 5 trên 10 ở chế độ `fixed`, và từ
  1/5 lên 5/5 ở chế độ `time`; `tao_bon_mo` từ 1 / 4 / 5 lên 1 / 10 / 10. Lý do: khi O thành bốn mở ở tầng 2, nút của X chỉ còn
  hai ô chặn nên giá trị thật của nhánh hiện ra sau ít lượt thăm.
- **Chi phí:** với cùng số mô phỏng, `tactical` chậm hơn khoảng 2,5 lần (1.000 mô phỏng: khoảng 0,5 giây so với 0,2 giây trên máy
  chạy) vì phải tìm ô thắng ngay ở mỗi nút và mỗi nước rollout. Số mô phỏng trung bình ở chế độ `time` cao hơn bản chuẩn chỉ vì
  các vị trí có nước bắt buộc: gốc chỉ còn một con là nút thắng, nên mỗi mô phỏng gần như không tốn chi phí.


In [24]:
tac_time_factory = lambda s: partial(MCTSAgent, seed=s, exploration=EXPLORATION, radius=RADIUS,
                                     time_margin_s=TIME_MARGIN_S, variant='tactical')
tac_fixed_factory = lambda s: partial(MCTSAgent, seed=s, exploration=EXPLORATION, radius=RADIUS, variant='tactical')
std_fixed_factory = lambda s: partial(MCTSAgent, seed=2000 + s, exploration=EXPLORATION, radius=RADIUS)
std_time_factory = lambda s: partial(MCTSAgent, seed=2000 + s, exploration=EXPLORATION, radius=RADIUS,
                                     time_margin_s=TIME_MARGIN_S)
H2H_TIME_SEEDS = list(range(5))     # đối đầu chế độ time tốn thời gian hơn: 5 cặp seed = 10 ván

start = time.perf_counter()
series = {}
series['tactical – Ngẫu nhiên (time)'] = run_series('MCTS tactical', tac_time_factory, 'Ngẫu nhiên', random_factory,
                                                   BUDGET_TIME, MATCH_SEEDS) + (BUDGET_TIME, MATCH_SEEDS)
series['tactical – standard (fixed)'] = run_series('MCTS tactical', tac_fixed_factory, 'MCTS standard', std_fixed_factory,
                                                  BUDGET_FIXED, MATCH_SEEDS) + (BUDGET_FIXED, MATCH_SEEDS)
series['tactical – standard (time)'] = run_series('MCTS tactical', tac_time_factory, 'MCTS standard', std_time_factory,
                                                 BUDGET_TIME, H2H_TIME_SEEDS) + (BUDGET_TIME, H2H_TIME_SEEDS)
print(f'Thời gian chạy thi đấu biến thể: {time.perf_counter() - start:.0f} giây')
rows = []
for name, (records, games, a_logs, budget, seeds) in series.items():
    check_series(records, a_logs, budget, seeds)
    rows.append({'trận': name, **series_summary('MCTS tactical', records, a_logs, budget)})
left(pd.DataFrame(rows).drop(columns=['cấu hình']))

Thời gian chạy thi đấu biến thể: 732 giây


,trận,chế độ,số ván,thắng,hòa,thua,điểm,lỗi/timeout,số lượt,độ trễ TB (s),độ trễ p95 (s),mô phỏng/lượt TB,mô phỏng/lượt min–max
0,tactical – Ngẫu nhiên (time),time,20,20,0,0,1.000,0,151,1.901,1.901,52173,1244–476591
1,tactical – standard (fixed),fixed,20,17,0,3,0.850,0,166,0.859,1.199,1000,1000–1000
2,tactical – standard (time),time,10,5,0,5,0.500,0,62,1.900,1.901,59933,1615–511188


**Quan sát:**
- **Với Random:** `tactical` cũng thắng 20/20 ở chế độ `time`, không timeout; độ trễ p95 vẫn khoảng 1,9 giây.
- **Cùng số mô phỏng** (`fixed` 1.000, 20 ván): `tactical` thắng **17/20** bản chuẩn (8/10 khi cầm X, 9/10 khi cầm O).
- **Cùng thời gian** (`time` 2,0 giây, 10 ván): `tactical` chỉ thắng **5/10** (2/5 khi cầm X, 3/5 khi cầm O), tức ngang bản chuẩn.
  Trong cùng 2 giây, mỗi mô phỏng của `tactical` đắt hơn khoảng 2,5 lần, nên ở các lượt không có nước bắt buộc nó chạy ít mô phỏng
  hơn bản chuẩn; lợi thế về luật chiến thuật bị bù lại bởi cây nông hơn.
- Mẫu 10 ván ở chế độ `time` quá nhỏ để kết luận bên nào mạnh hơn: kết quả này phụ thuộc tốc độ máy và có thể khác giữa các lần
  chạy. Nó chỉ cho thấy lợi thế 17/20 ở cùng số mô phỏng **không** tự động chuyển thành lợi thế khi cùng thời gian.


In [25]:
# Chi tiết các ván đối đầu tactical – standard (góc nhìn tactical); không bỏ ván nào.
left(pd.concat([series[k][1].assign(**{'trận': k}) for k in ['tactical – standard (fixed)', 'tactical – standard (time)']],
               ignore_index=True)[['trận', 'seed', 'MCTS tactical cầm', 'số nước', 'kết quả', 'lý do']])

,trận,seed,MCTS tactical cầm,số nước,kết quả,lý do
0,tactical – standard (fixed),0,X,14,thua,completed
1,tactical – standard (fixed),0,O,16,thắng,completed
2,tactical – standard (fixed),1,X,13,thắng,completed
3,tactical – standard (fixed),1,O,22,thắng,completed
4,tactical – standard (fixed),2,X,11,thắng,completed
5,tactical – standard (fixed),2,O,21,thua,completed
6,tactical – standard (fixed),3,X,15,thắng,completed
7,tactical – standard (fixed),3,O,12,thắng,completed
8,tactical – standard (fixed),4,X,19,thắng,completed
9,tactical – standard (fixed),4,O,18,thắng,completed


Không ván nào bị loại; mọi ván kết thúc `completed`. Các ván thua của `tactical` được giữ trong bảng để đối chiếu với log
(`series[...]`), phục vụ phân tích ở giải đấu chính thức.


## 10. Tổng kết và hướng dẫn tái lập

**Trả lời câu hỏi nghiên cứu (mục 1):**
- Chất lượng nước đi tăng rõ theo số mô phỏng: trên 5 vị trí có đáp án, đúng 3/50 → 25/50 → 30/50 ở 100 / 500 / 1.000 mô phỏng
  và 21/25 ở chế độ `time` 2,0 giây. Chi phí ở chế độ `fixed` tăng gần tuyến tính; chế độ `time` chạy khoảng 7.000–8.000 mô phỏng ở
  vị trí thường trên máy ghi ở mục 2.
- Chế độ `time` đáp ứng hạn cứng trong thực tế: các lượt dừng ở khoảng 1,9 giây, 0 timeout trong 90 ván qua runner (mục 8–9);
  riêng khảo sát mục 7 có 1 lần chạm mốc 2,0 giây, cho thấy biên 0,1 giây không phải bảo đảm tuyệt đối.
- Điểm yếu chính của bản chuẩn là đe dọa cần nhìn trước hai nước (chặn ba mở 1/5 ở chế độ `time`), do rollout ngẫu nhiên và
  lượt thăm chia đều cho khoảng 50 ứng viên mỗi tầng.
- Biến thể `tactical` sửa được các lỗi một nước và cải thiện rõ các thế hai nước; thắng bản chuẩn 17/20 khi cùng số mô phỏng,
  nhưng khi cùng 2 giây chỉ thắng 5/10 trong mẫu nhỏ.

**Cấu hình đề xuất bàn giao:** `MCTSAgent(variant='standard', exploration=√2, radius=2, time_margin_s=0.1)` với
`SearchBudget(mode='time', time_limit_s=2.0)`. Lý do: đã kiểm chứng 0 timeout, thắng 20/20 với Random, và trong đối đầu cùng thời
gian chưa có bằng chứng `tactical` mạnh hơn. `tactical` được giữ như **biến thể có tên** để đánh giá ở giải đấu chính thức
(≥ 100 ván/cặp); nếu giải đấu dùng ngân sách `fixed` thì `tactical` là lựa chọn tốt hơn theo số liệu mục 9.

**Hạn chế đã biết:** biên dừng 0,1 giây không loại bỏ hoàn toàn khả năng một lượt chạm 2,0 giây (1 lần ở mục 7); đối đầu chế độ
`time` chỉ 10 ván; luật chiến thuật của `tactical` chỉ nhìn một nước.

**Chưa đánh giá:** hệ số khám phá c và bán kính ứng viên (giữ cố định ở √2 và 2); tái sử dụng cây giữa các nước; đối thủ khác
Random (Minimax, Rule-based, Neural); giải đấu ≥ 100 ván/cặp và nhiều seed; giới hạn thời gian khác 2,0 giây; máy khác CPU ở mục 2.

**Hướng dẫn tái lập:**
1. Kích hoạt `.venv`, mở notebook từ thư mục `caro-11x11` hoặc `notebooks`, chạy **Restart Kernel & Run All**. Các mục phụ thuộc
   biến của mục trước (mục 7 dùng `TACTICS` ở mục 6) nên phải chạy theo thứ tự.
2. Thời gian chạy khoảng 20–25 phút; nên để máy rảnh trong lúc chạy vì chế độ `time` đo theo thời gian thật.
3. Kết quả chế độ `fixed` (mục 6–7, mục 8 `fixed`, đối đầu `fixed` ở mục 9) cố định theo seed. Số mô phỏng, độ trễ và kết quả các ván
   ở chế độ `time` phụ thuộc tốc độ máy nên có thể khác; các ô có `assert` kiểm tra toàn vẹn số ván, chế độ và giới hạn.
4. Seed: mục 7 dùng 0–9 (`fixed`) và 0–4 (`time`); mục 8–9 dùng MCTS seed `s`, Random seed `1000 + s`, bản chuẩn trong đối đầu
   seed `2000 + s`, với `s` = 0–9 (đối đầu chế độ `time`: 0–4).
